## Notebook 02: Data Preparation

### Objective
Prepare the MBDD2025 dataset for reproducible image-level, 5-class structural damage classification using PyTorch.

### Dataset
- Total selected images: 12,064
- Training images: 9,647
- Validation images: 1,214
- Test images: 1,203
- Image dimensions: 224 × 224 pixels
- Input channels: 3 (RGB)

### Damage Classes
1. Abscission
2. Bulge
3. Corrosion
4. Crack
5. Leakage

### Preprocessing
- Resized images to 224 × 224.
- Converted images to RGB tensors.
- Normalized using ImageNet mean and standard deviation.
- Applied random horizontal flipping and rotation to training images only.
- Kept validation and test transformations deterministic.

### Implementation
- Implemented a custom PyTorch Dataset that reads images directly from the ZIP archive.
- Created separate training, validation, and test datasets.
- Configured DataLoaders with a batch size of 32.
- Verified tensor shapes, data types, finite values, class mapping, and dataset lengths.

### Outcome
The data preparation pipeline passed its consistency checks and is ready for model development.

### Next Step
Implement and train a custom CNN baseline, followed by pretrained ResNet18, using the same data splits for a fair comparison.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

In [3]:
PROJECT_DIR = Path("/content/drive/MyDrive/DistributedAIProject")

In [4]:
# Dataset archive and validated manifest
ARCHIVE_PATH = PROJECT_DIR / "MBDD2025.zip"
MANIFEST_PATH = PROJECT_DIR / "mbdd_classification_manifest_deduplicated.csv"

In [5]:
# Checking that our existing files are available
print("Project directory exists: ", PROJECT_DIR.exists())
print("Dataset ZIP exists: ", ARCHIVE_PATH.exists())
print("Validated manifest exists: ", MANIFEST_PATH.exists())

assert ARCHIVE_PATH.exists(), "Dataset ZIP not found in Google Drive."
assert MANIFEST_PATH.exists(), "Validated manifest not found in Google Drive."

print("\nSetup successful!")

Project directory exists:  True
Dataset ZIP exists:  True
Validated manifest exists:  True

Setup successful!


In [6]:
import pandas as pd

# Loading and inspecting the manifest
manifest = pd.read_csv(MANIFEST_PATH)

print("Manifest shape: ", manifest.shape)
print("\nColumns: ", manifest.columns.tolist())

print("\nFirst 5 rows: ")
display(manifest.head())

print("\nImages per split: ")
print(manifest["split"].value_counts())

print("\nClass distribution per split: ")
display(pd.crosstab(manifest["label"], manifest["split"]))

print("\nMissing values: ")
print(manifest.isnull().sum())

print("\nUnique image filenames: ", manifest["image"].nunique())

Manifest shape:  (12064, 3)

Columns:  ['image', 'label', 'split']

First 5 rows: 


,image,label,split
0,Hefei1750.jpg,crack,train
1,Hefei3568.jpg,crack,test
2,Hefei206.jpg,corrosion,train
3,Hefei7694.jpg,crack,train
4,Hefei2910.jpg,leakage,train



Images per split: 
split
train         9647
validation    1214
test          1203
Name: count, dtype: int64

Class distribution per split: 


split,test,train,validation
label,,,
abscission,334,2672,334
bulge,71,597,82
corrosion,166,1329,167
crack,437,3490,436
leakage,195,1559,195



Missing values: 
image    0
label    0
split    0
dtype: int64

Unique image filenames:  12064


In [7]:
# Verifying image files inside the ZIP
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE_PATH, "r") as f:
  archiveNames = set(f.namelist())

  # Matching manifest filenames to the dataset's JPEGImages directory
  manifest["archive_path"] = manifest["image"].apply(lambda name: f"MBDD2025/JPEGImages/{name}")
  manifest["image_exists"] = manifest["archive_path"].isin(archiveNames)
  missingImages = manifest.loc[~manifest["image_exists"], ["image", "archive_path"]]

print("Manifest images: ", len(manifest))
print("Images found in ZIP: ", int(manifest["image_exists"].sum()))
print("Missing images: ", len(missingImages))

if len(missingImages):
    print("\nFirst missing images:")
    display(missingImages.head(10))

else:
    print("\nAll manifest images exist in the ZIP!")

assert len(missingImages) == 0, "Some manifest images are missing from the ZIP."

Manifest images:  12064
Images found in ZIP:  12064
Missing images:  0

All manifest images exist in the ZIP!


In [8]:
# Setting up image preprocessing
import torch

from torchvision import transforms
from PIL import Image
from io import BytesIO

In [9]:
# Reproducibility
random_seed = 42
torch.manual_seed(random_seed)

In [15]:
# Image configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32

In [16]:
# Fixed class-to-index mapping for all models
CLASS_NAMES = ["abscission", "bulge", "corrosion", "crack", "leakage",]
CLASS_TO_IDX = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

In [17]:
# ImageNet normalization for compatibility with pretrained ResNet18
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [18]:
# Training: augmentation + normalization
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [19]:
# Validation and test: deterministic preprocessing only
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("PyTorch version: ", torch.__version__)
print("Image size: ", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Batch size: ", BATCH_SIZE)
print("Class mapping: ", CLASS_TO_IDX)
print("Training augmentation: enabled")
print("Validation/test augmentation: disabled")

PyTorch version:  2.11.0+cu130
Image size:  224 x 224
Batch size:  32
Class mapping:  {'abscission': 0, 'bulge': 1, 'corrosion': 2, 'crack': 3, 'leakage': 4}
Training augmentation: enabled
Validation/test augmentation: disabled


In [20]:
# Creating the custom Dataset class
import zipfile

from torch.utils.data import Dataset
from io import BytesIO
from PIL import Image

class MBDD2025Dataset(Dataset):
  def __init__(self, dataframe, archive_path, transform=None):
    self.dataframe = dataframe.reset_index(drop=True).copy()
    self.archive_path = str(archive_path)
    self.transform = transform
    self.zip_file = None

  def __len__(self):
    return len(self.dataframe)

  def __getitem__(self, index):
    row = self.dataframe.iloc[index]
    image_name = row["image"]
    label = CLASS_TO_IDX[row["label"]]

    if self.zip_file is None:
      self.zip_file = zipfile.ZipFile(self.archive_path, "r")

    image_path = f"MBDD2025/JPEGImages/{image_name}"

    with self.zip_file.open(image_path) as f:
      image = Image.open(BytesIO(f.read())).convert("RGB")

    if self.transform is not None:
      image = self.transform(image)

    return image, label

  def __del__(self):
    if self.zip_file is not None:
      self.zip_file.close()

In [21]:
# Testing one image
train = manifest[manifest["split"]=="train"].copy()
sample_dataset = MBDD2025Dataset(
    dataframe=train.head(1),
    archive_path=ARCHIVE_PATH,
    transform=train_transform
)

imageTensor, labelIndex = sample_dataset[0]

print("Image tensor shape: ", imageTensor.shape)
print("Label index: ", labelIndex)
print("Label name: ", CLASS_NAMES[labelIndex])
print("Tensor dtype: ", imageTensor.dtype)
print("Tensor finite: ", torch.isfinite(imageTensor).all().item())

assert imageTensor.shape == (3, 224, 224)
assert labelIndex in range(len(CLASS_NAMES))
assert torch.isfinite(imageTensor).all()

print("\nSingle-image test passed!")

Image tensor shape:  torch.Size([3, 224, 224])
Label index:  3
Label name:  crack
Tensor dtype:  torch.float32
Tensor finite:  True

Single-image test passed!


In [22]:
# Create train, validation and test datasets
train = manifest[manifest["split"]=="train"].copy()
validation = manifest[manifest["split"]=="validation"].copy()
test = manifest[manifest["split"]=="test"].copy()

In [23]:
# Create datasets with the appropriate transformations
train_dataset = MBDD2025Dataset(
    dataframe=train,
    archive_path=ARCHIVE_PATH,
    transform=train_transform
)

validation_dataset = MBDD2025Dataset(
    dataframe=validation,
    archive_path=ARCHIVE_PATH,
    transform=eval_transform
)

test_dataset = MBDD2025Dataset(
    dataframe=test,
    archive_path=ARCHIVE_PATH,
    transform=eval_transform
)

print("Training dataset: ", len(train_dataset))
print("Validation dataset: ", len(validation_dataset))
print("Test dataset: ", len(test_dataset))
print("Total images: ", len(train_dataset)+len(validation_dataset)+len(test_dataset))

assert len(train_dataset) == 9647
assert len(validation_dataset) == 1214
assert len(test_dataset) == 1203

print("\nAll three datasets created successfully!")

Training dataset:  9647
Validation dataset:  1214
Test dataset:  1203
Total images:  12064

All three datasets created successfully!


In [24]:
# Creating PyTorch DataLoaders
from torch.utils.data import DataLoader

# Reproducible shuffling for training
def seed_worker(worker_id):
  import random
  import numpy as np

  worker_seed = torch.initial_seed()%(2**32)
  np.random.seed(worker_seed)
  random.seed(worker_seed)

generator = torch.Generator()
generator.manual_seed(random_seed)

# Using zero workers for compatibility with ZIP-backed datasets in Colab
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)
print("DataLoader setup successful!")

Training batches: 302
Validation batches: 38
Test batches: 38
Batch size: 32
DataLoader setup successful!


In [26]:
# Fetching one batch from the training DataLoader
images, labels = next(iter(train_loader))

print("Images shape: ", images.shape)
print("Labels shape: ", labels.shape)
print("Image dtype: ", images.dtype)
print("Label dtype: ", labels.dtype)
print("Pixel values finite: ", torch.isfinite(images).all().item())

print("\nLabels in this batch: ", labels.tolist())

print("\nClass names represented in this batch: ", sorted({CLASS_NAMES[i] for i in labels.tolist()}))

# Assertions
assert images.ndim == 4
assert images.shape[1:] == (3, 224, 224)
assert labels.ndim == 1
assert labels.dtype == torch.int64
assert labels.min().item() >= 0
assert labels.max().item() < len(CLASS_NAMES)
assert torch.isfinite(images).all()

print("\nBatch validation passed!")

Images shape:  torch.Size([32, 3, 224, 224])
Labels shape:  torch.Size([32])
Image dtype:  torch.float32
Label dtype:  torch.int64
Pixel values finite:  True

Labels in this batch:  [0, 4, 3, 3, 0, 1, 3, 1, 4, 4, 4, 2, 0, 0, 4, 3, 0, 2, 3, 0, 1, 4, 4, 3, 3, 3, 0, 0, 0, 0, 0, 0]

Class names represented in this batch:  ['abscission', 'bulge', 'corrosion', 'crack', 'leakage']

Batch validation passed!


In [27]:
# Final dataset consistency checks
assert CLASS_NAMES == ["abscission", "bulge", "corrosion", "crack", "leakage"]

assert CLASS_TO_IDX == {
    "abscission": 0,
    "bulge": 1,
    "corrosion": 2,
    "crack": 3,
    "leakage": 4
}

In [28]:
# Verifying dataset sizes
assert len(train_dataset) == 9647
assert len(validation_dataset) == 1214
assert len(test_dataset) == 1203

In [30]:
# Verifying that evaluation transaforms are deterministic
sample_row = validation.iloc[0]
sample_dataset = MBDD2025Dataset(dataframe=validation.head(1), archive_path=ARCHIVE_PATH, transform=eval_transform)

image_a, label_a = sample_dataset[0]
image_b, label_b = sample_dataset[0]

assert torch.equal(image_a, image_b)
assert label_a == label_b

print("Class mapping: verified")
print("Dataset lengths: verified")
print("Evaluation transforms: deterministic")
print("Input shape: ", tuple(image_a.shape))
print("All final consistency checks passed!")

Class mapping: verified
Dataset lengths: verified
Evaluation transforms: deterministic
Input shape:  (3, 224, 224)
All final consistency checks passed!
